# RFM Feature Engineering

This notebook computes Recency, Frequency, and Monetary (RFM) metrics from the cleaned retail dataset.
Feature engineering is performed in both SQL (canonical) and Pandas (validation), with quantile-scored RFM features saved for segmentation modeling.

---

RFM Metric Definitions:
- **Recency** – Days since the customer's last purchase (lower is better)
- **Frequency** – Total number of unique invoices per customer
- **Monetary** – Total revenue generated per customer (quantity × price)
- **R / F / M Scores** – Quartile-based scores (1–4) assigned via NTILE in SQL or `qcut`/`rank` in Pandas
- **RFM Score** – Composite score combining individual R, F, and M scores

In [1]:
import pandas as pd
import os
from sqlalchemy import create_engine, text

# Local src

from src.utils.eda_utils import df_overview
from src.tasks.pd_feature_engineering import pd_feature_engineering
from src.pipelines.pipeline import feature_engineering_pipeline

from src.config.config import DB_CREATED
from src.config.paths import PROCESSED_DIR

print('Done')


Done


In [2]:
# the SQL version of feature engineering in this project.

feature_engineering_pipeline()

engine = create_engine(DB_CREATED)

df_sql_featured = pd.read_sql(text("SELECT * FROM mart.mart_customer_rfm_features"), engine)
print(f"df_rfm loaded: {df_sql_featured.shape[0]} rows, {df_sql_featured.shape[1]} columns")
df_sql_featured.set_index('customer_id', inplace=True)
df_sql_featured.sort_index(inplace=True)
df_sql_featured


2026-04-26 14:16:14 | INFO    | prefect | Starting temporary server on http://127.0.0.1:8508
See https://docs.prefect.io/v3/concepts/server#how-to-guides for more information on running a dedicated Prefect server.
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: GET http://127.0.0.1:8508/api/health "HTTP/1.1 200 OK"
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: GET http://127.0.0.1:8508/api/admin/version "HTTP/1.1 200 OK"
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: GET http://127.0.0.1:8508/api/csrf-token?client=1a860c65-9704-4922-b493-668f6e9e864d "HTTP/1.1 422 Unprocessable Entity"
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: POST http://127.0.0.1:8508/api/flows/ "HTTP/1.1 200 OK"
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: POST http://127.0.0.1:8508/api/flow_runs/ "HTTP/1.1 201 Created"
2026-04-26 14:16:20 | INFO    | httpx | HTTP Request: POST http://127.0.0.1:8508/api/flow_runs/4ac697fe-87fa-4c33-89a7-015158e87a10/set_state "HTTP/1.1 201 C

df_rfm loaded: 5847 rows, 11 columns


,recency,frequency,monetary,recency_score,frequency_score,monetary_score,r_f_score,segment,return_ratio,avg_order_value
customer_id,,,,,,,,,,
12346,325,14,368.36,2,5,2,25,cant_lose,0.995250,26.311429
12347,1,8,4921.53,5,4,5,54,champions,0.000000,615.191250
12348,74,5,1658.40,3,3,4,33,need_attention,0.000000,331.680000
12349,18,4,3654.54,4,3,5,43,potential_loyalists,0.006565,913.635000
12350,309,1,294.40,2,1,2,21,hibernating,0.000000,294.400000
...,...,...,...,...,...,...,...,...,...,...
18283,3,22,2658.95,5,5,4,55,champions,0.000000,120.861364
18284,431,1,411.68,1,2,2,12,hibernating,0.000000,411.680000
18285,660,1,377.00,1,2,2,12,hibernating,0.000000,377.000000


In [3]:
'''
Pandas feature engineering CAN be used for testing purposes, but not implemented in this project.
The SQL version is the main implementation for feature engineering in this project.
if you have any issues with the SQL feature engineering, I built the pandas version for feature engineering, 
but comment out the SQL version and further testing code to avoid broken logic.
'''
path_base_csv = PROCESSED_DIR / 'base_retail.csv'
df = pd.read_csv(path_base_csv)
df['invoice_date'] = pd.to_datetime(df['invoice_date'])
df_pd_featured = pd_feature_engineering(df)
print(f"df_rfm loaded: {df_pd_featured.shape[0]} rows, {df_pd_featured.shape[1]} columns")
df_pd_featured.sort_index(inplace=True)
df_pd_featured

2026-04-26 14:16:43 | INFO    | src.tasks.pd_feature_engineering | === PANDAS FEATURE ENGINEERING STARTED ===
2026-04-26 14:16:44 | INFO    | src.tasks.pd_feature_engineering | === PANDAS FEATURE ENGINEERING COMPLETED SUCCESSFULLY ===


df_rfm loaded: 5847 rows, 10 columns


,recency,frequency,monetary,recency_score,frequency_score,monetary_score,r_f_score,segment,return_ratio,avg_order_value
customer_id,,,,,,,,,,
12346,325,14,368.36,2,5,2,25,cant_lose,0.995250,26.311429
12347,1,8,4921.53,5,4,5,54,champions,0.000000,615.191250
12348,74,5,1658.40,3,3,4,33,need_attention,0.000000,331.680000
12349,18,4,3654.54,4,3,5,43,potential_loyalists,0.006565,913.635000
12350,309,1,294.40,2,1,2,21,hibernating,0.000000,294.400000
...,...,...,...,...,...,...,...,...,...,...
18283,3,22,2658.95,5,5,4,55,champions,0.000000,120.861364
18284,431,1,411.68,1,2,2,12,hibernating,0.000000,411.680000
18285,660,1,377.00,1,2,2,12,hibernating,0.000000,377.000000


In [4]:
df_overview(df_sql_featured)

================================= Shape =================================
(5847, 10)
================================= Info =================================
<class 'pandas.core.frame.DataFrame'>
Index: 5847 entries, 12346 to 18287
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          5847 non-null   int64  
 1   frequency        5847 non-null   int64  
 2   monetary         5847 non-null   float64
 3   recency_score    5847 non-null   int64  
 4   frequency_score  5847 non-null   int64  
 5   monetary_score   5847 non-null   int64  
 6   r_f_score        5847 non-null   object 
 7   segment          5847 non-null   object 
 8   return_ratio     5847 non-null   float64
 9   avg_order_value  5847 non-null   float64
dtypes: float64(3), int64(5), object(2)
memory usage: 502.5+ KB
None
================================= Columns =================================
Index(['recency', 'frequency', 'moneta

- 5847 customers for the analysis and modeling;
- no NaNs;
- correct dtypes;
- right features;
- no unexpected values for rfm calc;

In [5]:
df_overview(df_pd_featured)

================================= Shape =================================
(5847, 10)
================================= Info =================================
<class 'pandas.core.frame.DataFrame'>
Index: 5847 entries, 12346 to 18287
Data columns (total 10 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   recency          5847 non-null   int64  
 1   frequency        5847 non-null   int64  
 2   monetary         5847 non-null   float64
 3   recency_score    5847 non-null   int64  
 4   frequency_score  5847 non-null   int64  
 5   monetary_score   5847 non-null   int64  
 6   r_f_score        5847 non-null   object 
 7   segment          5847 non-null   object 
 8   return_ratio     5847 non-null   float64
 9   avg_order_value  5847 non-null   float64
dtypes: float64(3), int64(5), object(2)
memory usage: 502.5+ KB
None
================================= Columns =================================
Index(['recency', 'frequency', 'moneta

- some columns values are visually shown in exponent;
- r_f_score distribution is identical;
- segment destribution differs slightly;

In [6]:
# Testing dataframes on equality
# pd.testing.assert_frame_equal(df_pd_featured, df_sql_featured, check_dtype=False)

- mismatch is small, less than 0.1% (0.05131%), the calculated data is almost identical;
- as I found, pandas qcut and rank have slighly different logic than NTILE in SQL, it probably causes this behavior;
- it is a better approach to do a full test to find out the exact causes, but the pandas version is here not primarily for testing purposes;

In [7]:
pd_to_csv_path = os.path.join('..', 'data', '03_featured', 'featured_retail.csv')

os.makedirs(os.path.dirname(pd_to_csv_path), exist_ok=True)
df_sql_featured.to_csv(pd_to_csv_path)

print(f"Saved to: {pd_to_csv_path}")

Saved to: ..\data\03_featured\featured_retail.csv


- the SQL version is used as canonical and saved as featured csv;

## Summary of RFM Feature Engineering

### Approach:
- RFM metrics calculated using both **SQL (canonical)** and **Pandas (additional)**
- Return ratio and average order value are calculated for better segmentation and analysis to support RFM scores.

### Key Outputs:
- **5,847 customers** retained after cleaning, with no missing values and correct dtypes
- SQL and Pandas implementations produce near-identical results (**< 0.1% mismatch**, ~0.051%)
- Minor differences may be due to logic difference between NTILE and Pandas `qcut`

### Next Steps:
The featured dataset has been saved to `../data/03_featured/featured_rfm_retail.csv` and is ready for:
1. Post-feature EDA (RFM distributions, segment profiling, correlation analysis)
2. Customer segmentation modeling (clustering)